<a href="https://colab.research.google.com/github/GiuseppeRizzello/Profession-AI-Projects/blob/main/RizzelloPythonProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Store management software

The goal is to create a text-based Python program, executable from the command line, that manages the shop's inventory, sales, and profits. The project assesses the ability to translate specifications into code while adhering strictly to them, to select appropriate data structures, and to write clean, documented, and robust code.

Before writing any code, consider which data structures to use (lists, tuples, dictionaries, or combinations such as lists of dictionaries) and how to organize the functionality into dedicated functions.

In [ ]:
import json
from os import path

# Check that the inventory file exists and is not empty.
if path.isfile('inventory.json') and path.getsize('inventory.json') != 0:
    with open('inventory.json', 'r', encoding='utf-8') as inventory_file:
        product_info = json.load(inventory_file)
else:
    # If the inventory file does not exist or is empty, create an empty
    # dictionary in which each product name corresponds to a dictionary
    # containing quantity and prices.
    product_info = {}

# Dictionary containing the products and the quantities sold during the session.
session_sales = {}

# Help string listing the available commands.
help_string = 'The available commands are:\n'
help_string += 'add: add a product to the inventory\n'
help_string += 'list: list the products in the inventory\n'
help_string += 'sell: register a sale\n'
help_string += 'profits: show total profits\n'
help_string += 'help: show the available commands\n'
help_string += 'exit: exit the program'


def is_type(value, data_type):
    """
    value (str): string to check.
    data_type: int or float.

    Returns True if data_type accepts value as an argument, False if it raises
    an exception.
    """
    try:
        value = data_type(value)
        return True
    except ValueError:
        return False


def input_control(prompt, data_type):
    """
    prompt (str): specifies the input request.
    data_type: int or float.

    Repeats the input request until the entered value is a positive integer
    (if data_type = int) or a positive decimal number (if data_type = float).

    Returns the numeric input value.
    """
    no_error = False

    while not no_error:
        try:
            value = input(prompt)

            # Replace commas with dots because some users may be accustomed
            # to using commas as decimal separators.
            if data_type == float and ',' in value:
                value = value.replace(',', '.')

            if not is_type(value, data_type):
                if data_type == float:
                    raise ValueError('Error: non-numeric value')
                elif data_type == int:
                    raise ValueError('Error: the value is not an integer')

            value = data_type(value)

            if value < 0:
                raise ValueError('Error: negative value.')

            no_error = True
        except ValueError as e:
            print(e)

    return value


def help():
    """
    Prints the help string.
    """
    print(help_string)


def add(product_info):
    """
    product_info (dict): dictionary of products in the inventory.

    Asks which product to add to the inventory.
    If it is already present, only the quantity is updated.
    If it is not present, the purchase and selling prices are also requested.
    Finally, prints the product and the quantity added.

    Returns the updated dictionary.
    """
    name = input('Product name: ')
    quantity = input_control('Quantity: ', int)

    if quantity != 0:
        # Consider the case in which the product is not already in the inventory
        if name not in product_info:
            purchase_price = input_control('Purchase price: ', float)
            selling_price = input_control('Selling price: ', float)

            product_info[name] = {
                'quantity': quantity,
                'purchase price': purchase_price,
                'selling price': selling_price
            }

        # Consider the case in which the product is already in the inventory.
        else:
            product_info[name]['quantity'] += quantity

        with open('inventory.json', 'w', encoding='utf-8') as inventory_file:
            json.dump(product_info, inventory_file, indent=4, ensure_ascii=False)

        print(f'ADDED: {quantity} X {name}')

    return product_info


def list_products(product_info):
    """
    product_info (dict): dictionary of products in the inventory.

    Prints a list of the products in the inventory, together with their quantity
    and selling price.
    """
    if not product_info:
        print('The inventory is empty.')
    else:
        # To align the columns properly, adjust the width to the length
        # of the product names.
        width = max(len(name) for name in product_info) + 2

        # If all product names are short enough, set a standard width.
        if width < 10:
            width = 10

        print(f"{'PRODUCT':<{width}} {'QUANTITY':<10} {'PRICE':<10}")

        for name in product_info:
            print(
                f"{name:<{width}} {product_info[name]['quantity']:<10}",
                f"€{product_info[name]['selling price']:<8}"
            )


def sell(product_info, session_sales):
    """
    product_info (dict): dictionary of products in the inventory.
    session_sales (dict): dictionary of products sold since the beginning of the
    program.

    Asks which products to sell and their quantity, and removes that quantity
    from the products in the inventory.
    Finally, prints the inputs together with the total revenue of this call to
    the function.

    Returns the updated inventory and sales dictionaries.
    """
    total = 0
    # Dictionary of products sold during the current call to the sell function.
    partial_sales = {}

    # Variable indicating whether a sale actually took place, since entering
    # 0 as the quantity does not actually sell any product.
    real_sale = False

    answer = 'yes'

    while answer == 'yes':
        try:
            name = input('Product name: ')

            if name not in product_info:
                raise KeyError(
                    'Error: you cannot sell a product that is not in the inventory'
                )

            quantity_sold = input_control('Quantity: ', int)

            if quantity_sold > product_info[name]['quantity']:
                raise ValueError(
                    'Error: the quantity is greater than the available quantity.'
                )

        except KeyError as e:
            # Use args because KeyError prints the message in quotation marks.
            print(e.args[0])

            # Exit the while loop so that the following code is not executed,
            # allowing the user to use the list command and check the inventory.
            break

        except ValueError as e:
            print(e)

            # Same as above: return to the main menu so that the user can check
            # the inventory before trying another sale.
            break

        if quantity_sold != 0:
            real_sale = True

            product_info[name]['quantity'] -= quantity_sold
            total += quantity_sold * product_info[name]['selling price']

            # Update the quantity sold during the current call to the sell function.
            # Use get() so that a missing product starts with a quantity of 0.
            partial_sales[name] = partial_sales.get(name, 0) + quantity_sold

            # Update the quantity sold during the current program session.
            session_sales[name] = session_sales.get(name, 0) + quantity_sold

        answer = input('Add another product? (yes/no): ')
        # Use lower() to also accept answers such as Yes, No, or NO.
        answer = answer.lower()

        while answer != 'yes' and answer != 'no':
            answer = input('Add another product? (yes/no): ')
            answer = answer.lower()

    if real_sale:
        print('SALE REGISTERED')

        for name in partial_sales:
            print(
                f"- {partial_sales[name]} X {name}:",
                f"€{product_info[name]['selling price']}"
            )

        print(f'Total: €{total:.2f}')

        with open('inventory.json', 'w', encoding='utf-8') as inventory_file:
            json.dump(product_info, inventory_file, indent=4, ensure_ascii=False)

    return product_info, session_sales


def profits(product_info, session_sales):
    """
    product_info (dict): dictionary of products in the inventory.
    session_sales (dict): dictionary of products sold since the beginning of the
    program.

    Calculates the gross and net profits from the sales made since the beginning
    of the program and prints them to the screen.
    """
    gross = 0
    expenses = 0

    for name in session_sales:
        gross += session_sales[name] * product_info[name]['selling price']
        expenses += session_sales[name] * product_info[name]['purchase price']

    net = gross - expenses

    print(f'Profit: gross=€{gross:.2f} net=€{net:.2f}')


command = input('Enter a command: ')

while command != 'exit':
    if command == 'help':
        help()
    elif command == 'add':
        product_info = add(product_info)
    elif command == 'list':
        list_products(product_info)
    elif command == 'sell':
        product_info, session_sales = sell(product_info, session_sales)
    elif command == 'profits':
        profits(product_info, session_sales)
    else:
        print('Invalid command')
        help()

    command = input('Enter a command: ')


# If the session sales dictionary is not empty, save the sales to a JSON file.
if session_sales:
    total_sales = {}

    if path.isfile('sales.json') and path.getsize('sales.json') != 0:
        with open('sales.json', 'r', encoding='utf-8') as sales_file:
            total_sales = json.load(sales_file)

    for name in session_sales:
        total_sales[name] = total_sales.get(name, 0) + session_sales[name]

    with open('sales.json', 'w', encoding='utf-8') as sales_file:
        json.dump(total_sales, sales_file, indent=4, ensure_ascii=False)


# Delete products with zero quantity from the inventory for cleanup.
# Do this only at the end so that the user can still see the price information
# of the sold product while using the program.

product_info_keys = list(product_info.keys())

# We made a copy of the list of keys so that we do not encounter problems
# in the for loop, since we use del.

# Variable indicating whether at least one product has sold out.
sold_out = False

for name in product_info_keys:
    # If the product has been completely sold out.
    if product_info[name]['quantity'] == 0:
        del product_info[name]
        sold_out = True

# Save the inventory changes if necessary.
if sold_out:
    with open('inventory.json', 'w', encoding='utf-8') as inventory_file:
        json.dump(product_info, inventory_file, indent=4, ensure_ascii=False)

print('Bye bye')
